# Chapter 4 · Govern & Capture

Serving a model (Chapter 3) is table stakes. This chapter adds the layer most demos skip — the layer a **self-healing** system actually needs to be trusted:

1. **Capture** every inference in a UC Delta table (inference tables / AI Gateway).
2. **Inspect** the captured payloads.
3. **Monitor** the endpoint for drift / quality.
4. **Cost** — a per-decision economics ledger: open-source in-zone vs. frontier.

**Prerequisite:** the `otel-embedding-335m` serving endpoint from [Chapter 3](./03_productionize_on_databricks.ipynb).

> ⚠️ API surfaces evolve across Databricks SDK versions — cells flag where to double-check a call if it errors.

> OTel = **Open Telco**, not OpenTelemetry.

In [ ]:
%pip install -q "mlflow>=2.13" databricks-sdk
dbutils.library.restartPython()

In [ ]:
# --- Config: match Chapter 3 ---
CATALOG      = "main"
SCHEMA       = "otel_selfhealing"
EMB_ENDPOINT = "otel-embedding-335m"
TABLE_PREFIX = "otel_emb_payload"
LEDGER_TABLE = f"{CATALOG}.{SCHEMA}.otel_cost_ledger"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("config ok")

## 1. Turn on inference capture (AI Gateway)

Enable request/response logging + usage tracking on the endpoint. Every inference now lands in a governed UC Delta table — **this is the "capture model serving" deliverable.**

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayInferenceTableConfig, AiGatewayUsageTrackingConfig)

w = WorkspaceClient()
# If your SDK version names these differently, check: help(w.serving_endpoints.put_ai_gateway)
w.serving_endpoints.put_ai_gateway(
    name=EMB_ENDPOINT,
    inference_table_config=AiGatewayInferenceTableConfig(
        enabled=True, catalog_name=CATALOG, schema_name=SCHEMA, table_name_prefix=TABLE_PREFIX),
    usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
)
print("inference capture enabled on", EMB_ENDPOINT)

## 2. Send traffic, then inspect the captured payloads

Capture is asynchronous — payloads appear a short delay after traffic. The payload table is named from the prefix (often `<prefix>_payload`); discover the exact name with `SHOW TABLES`.

In [ ]:
import mlflow.deployments
client = mlflow.deployments.get_deploy_client("databricks")

queries = [
    "cells show low downlink throughput at low PRB load",
    "neighboring cells at 98% PRB in the busy hour",
    "two adjacent cells share the same PCI",
    "CPRI fronthaul link lost to the radio unit",
]
for q in queries:
    client.predict(endpoint=EMB_ENDPOINT, inputs={"inputs": [q]})
print(f"sent {len(queries)} requests — payloads will land shortly")

In [ ]:
# Find the payload table, then read it back.
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE '{TABLE_PREFIX}*'"))

In [ ]:
PAYLOAD_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_payload"   # adjust to the name shown above
display(spark.sql(f"SELECT * FROM {PAYLOAD_TABLE} ORDER BY 1 DESC LIMIT 20"))

## 3. Monitor the inference table (optional)

Attach **Lakehouse Monitoring** for drift / volume / quality dashboards. This is a time-series monitor over the captured payloads; the timestamp column depends on your payload schema (inspect it above first).

In [ ]:
# Optional — uncomment once you've confirmed the payload table + its timestamp column.
# from databricks.sdk.service.catalog import MonitorTimeSeries
# w.quality_monitors.create(
#     table_name=PAYLOAD_TABLE,
#     assets_dir=f"/Workspace/Users/{w.current_user.me().user_name}/otel_monitors",
#     output_schema_name=f"{CATALOG}.{SCHEMA}",
#     time_series=MonitorTimeSeries(timestamp_col="timestamp_ms", granularities=["1 hour"]),
# )
# print("monitor created — dashboard generates on first refresh")

## 4. Cost-economics ledger — open-source in-zone vs. frontier

For each grounded decision, record what the sovereign open-source OTel stack cost (~$0 marginal — flat-hosted, in-tenant) vs. what a frontier per-token model *would* have cost. This is the sovereignty + economics story a telecom operator cares about, captured as a governed Delta table.

In [ ]:
import pandas as pd, time

def ledger_row(workload, prompt_tokens, completion_tokens,
               frontier_in_per_mtok, frontier_out_per_mtok, chosen="otel-in-zone"):
    frontier = (prompt_tokens * frontier_in_per_mtok + completion_tokens * frontier_out_per_mtok) / 1e6
    return {"ts": time.time(), "workload": workload, "chosen": chosen,
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
            "opensource_usd": 0.0, "frontier_usd": round(frontier, 6),
            "saved_usd": round(frontier, 6)}

rows = [
    ledger_row("telco_rag_grounding", 1200, 400, frontier_in_per_mtok=3.0, frontier_out_per_mtok=15.0),
    ledger_row("telco_rag_grounding", 900,  300, frontier_in_per_mtok=3.0, frontier_out_per_mtok=15.0),
]
(spark.createDataFrame(pd.DataFrame(rows))
      .write.mode("append").saveAsTable(LEDGER_TABLE))
display(spark.sql(f"SELECT workload, count(*) decisions, round(sum(saved_usd),4) total_saved_usd "
                  f"FROM {LEDGER_TABLE} GROUP BY workload"))

## What you built — and the full picture

✅ Every inference is **captured** in a governed UC Delta table (audit trail).  
✅ The endpoint is **monitorable** for drift / quality.  
✅ Each decision has a **cost** — and the sovereign open-source path is quantifiably cheaper than frontier.

This is the governance-of-use layer that turns the OTel pipeline into something an operator can **trust, audit, and run** — the missing piece that makes a self-healing network responsible, not just automated.

**The full ladder:**

| Chapter | What runs | Where |
|---|---|---|
| 0 · Vision + Step 0 | one OTel model | laptop |
| 1 · RAG pipeline | embed → retrieve → rerank → ground → abstain | laptop |
| 2 · Agent loop | the pipeline inside the `otel.py` ReAct loop | laptop |
| 3 · Productionize | log → UC → serve → Vector Search | Databricks |
| **4 · Govern & capture** | **inference tables + monitoring + cost ledger** | Databricks |

From here, the north-star [`otel.py`](../otel.py) agent runs fully on governed, captured OTel intelligence — a self-healing loop a telecom operator can actually put near a live network.